# Mathematisches Modell: Hazmat-HVRP mit E-Fahrzeugen, EU-Lenkzeiten und Risiko-Kosten-Zeit-Zielfunktion


---

## 0. Einordnung und Problemklassifikation

Dieses Dokument beschreibt das formalisierte mathematische Modell für ein **Single-Trip Heterogeneous-Fleet Hazmat-VRP** mit E-Lkw-Reichweitenrestriktionen, Lade-Side-Trips und integrierten EU-Lenk- und Ruhezeiten. Es ist als gemischt-ganzzahliges lineares Programm (MILP) formuliert. 

Das Modell kombiniert das kapazitierte VRP (Toth & Vigo, 2002) mit heterogener Flotte (HFVRP; Golden et al., 1984) und integriert ein kantenadditives Hazmat-Risikoziel (Erkut & Verter, 1998; Holeczek, 2021). Hinzu kommen E-VRP-Ladelogiken (Schneider, Stenger & Goeke, 2014) sowie die Abbildung gesetzlicher Lenkzeiten (Goel, 2009; VO (EG) 561/2006).

Die Modellarchitektur ist zweistufig aufgebaut:
1. **Ebene 1 (Vorverarbeitung):** Reduktion des realen Straßengraphen auf OD-Matrizen (Distanz, Zeit, Risikoprämie) zwischen Depot, Kunden und Ladesäulen inklusive Profil-Vorauswahl.
2. **Ebene 2 (Routing):** Lösung des NP-schweren Optimierungsproblems auf dem reduzierten Graphen mittels MILP.


## Ebene 1a: Graphreduktion (Vorverarbeitung, unverändert im Prinzip)

**Eingabe:** realer Straßengraph $G_{road} = (N_{road}, E_{road})$ mit Distanz, Fahrzeit, Risikodaten, Tunnelkategorie (ADR) und Autobahnkennzeichnung je Kante.

Für jedes Knotenpaar $(i,j)$ der relevanten Knoten (Depot, Kunden, Ladesäulen) werden je **Profil** $\pi \in P = \{\text{shortest}, \text{fastest}, \text{safest}\}$ und je **Ladezustand** $\ell \in \{\text{loaded}, \text{empty}\}$ kürzeste Pfade per Dijkstra berechnet. Ergebnis je $(i,j,\pi,\ell)$: Distanz $d^{\pi,\ell}_{ij}$ [km], Fahrzeit $t^{\pi,\ell}_{ij}$ [min] und der Dijkstra-Zielwert $cost^{\pi,\ell}_{ij}$.

Das Risikomaß des safest-Profils bleibt das kantenadditive Modell:

$$
Risk_e = \mu_k \cdot \big(\alpha \cdot \text{PopDens}_e + \beta \cdot \text{AccRate}_e + \gamma \cdot \text{NatRes}_e\big), \qquad \alpha = \beta = 0{,}40,\; \gamma = 0{,}20
$$

mit der Gewichtung nach Cuneo et al. (2018) für die Kraftstofflogistik und Naturschutzflächen als drittem Faktor (methodische Erweiterung im Sinne von List & Mirchandani, 1988). Kantenadditive Risikomaße dieser Form sind der Standardansatz der Hazmat-Routing-Literatur (Erkut & Verter, 1998).

**Beladene Pfade (loaded):** Vor dem Dijkstra-Lauf werden alle für die Gefahrgutklasse gesperrten Tunnelkanten entfernt (Vorab-Screening nach List & Mirchandani, 1988); existiert für eine Relation nur ein Tunnelpfad, gilt $cost = \infty$. Zusätzlich wirkt der Autobahn-Penalty $\Pi_e$ (aktiv ab der Gewichtsschwelle $HwyThreshold_k \approx 0{,}25$ t für Klasse 3), sodass Autobahnabschnitte bevorzugt werden, wo Alternativen existieren.

**Leere Pfade (empty):** freie Routung ohne Tunnel-/Autobahnrestriktion und ohne Risikoterm. Da das Schadensrisiko eine Funktion der transportierten Menge ist, ist es bei Leerfahrt null (List & Mirchandani, 1988); die Unterscheidung lastabhängiger von lastunabhängigen Risikomodellen und ihr Einfluss auf die optimalen Touren ist empirisch belegt (Holeczek, 2021; Bula et al., 2016). Die ADR-Tunnelkategorien greifen nur im beladenen Zustand, konsistent zur Datenlage (empty-Relationen von C10 sind endlich).

Die Verwendung vorab berechneter Pfade/Distanzmatrizen als Fundament des VRP ist Standard (Toth & Vigo, 2002) und wird für das Hazmat-VRP explizit als Architektur „Pfadgenerierung → Routing" beschrieben (Holeczek, 2021).

---

## Ebene 1b: Risikoprämie und Profil-Vorauswahl (neu formalisiert, Code §2)

Der Solver wählt je gerichteter Relation $(i,j)$ **ein** Profil aus und übergibt nur diese eine Kante an das MILP. Formalisierung:

**(1) Risikoprämie in km-Äquivalenten.** Für beladene Relationen:

$$
\rho^{\pi}_{ij} = \max\big\{0,\; cost^{\pi}_{ij} - d^{\pi}_{ij}\big\},
\qquad
\rho^{\pi}_{ij} := \Lambda \;\; \text{falls } cost^{\pi}_{ij} = \infty,
\qquad
\Lambda = 2 \cdot \max_{(i,j,\pi):\, \rho < \infty} \rho^{\pi}_{ij}
$$

Relationen, deren einziger Pfad einen gesperrten Tunnel nutzt (ADR-Konflikt, vgl. §13: Kunde C10), erhalten also eine **prohibitive, aber endliche** Risikoprämie $\Lambda$, damit die Bedienpflicht erfüllbar bleibt und der Solver solche Traversen dennoch minimiert — genau die von Erkut, Tjandra & Verter (2007, Kap. 9.2) empfohlene Behandlung von Sperrungen im Planungsmodell.

**(2) Gewichtete Profilbewertung und Auswahl** (mit den globalen Gewichten $w_1, w_2, w_3$ und den Spaltenmaxima $\bar\rho, \bar d, \bar t$ als Normierung):

$$
eval^{\pi}_{ij} = w_1 \frac{\rho^{\pi}_{ij}}{\bar\rho} + w_2 \frac{d^{\pi}_{ij}}{\bar d} + w_3 \frac{t^{\pi}_{ij}}{\bar t},
\qquad
\pi^*(i,j) \in \arg\min_{\pi \in P} eval^{\pi}_{ij}
$$

Die Kantenparameter des MILP werden auf $d_{ij} := d^{\pi^*}_{ij}$, $t_{ij} := t^{\pi^*}_{ij}$, $\rho_{ij} := \rho^{\pi^*}_{ij}$ gesetzt.

- **Beladene Kanten** $A_{out}$ (Depot→Kunde, Kunde→Kunde): Auswahl wie oben.
- **Leerkanten** $A_{ret}$ (Kunde→Depot): $\rho_{i0} := 0$; Auswahl per $w_2 \frac{d}{\bar d} + w_3 \frac{t}{\bar t}$ (Distanz-Zeit-Trade-off, ohne Renormierung der zwei Gewichte  für die Argmin-Auswahl unerheblich).
- **Ladesäulen-Kanten** (Kunde↔Säule): identische Auswahl mit der in der Charger-Matrix vorhandenen, einheitenreinen `risk`-Spalte.

**Methodische Einordnung.** Die Zielfunktion in Ebene 2 ist kantenseparabel; bei gegebenen Gewichten ist die kantenweise Minimalwahl daher mit der gewichteten Gesamtsumme konsistent (Skalarisierung per gewichteter Summe: Marler & Arora, 2004). Die Auswahl aus einem kleinen Menü vordefinierter Alternativpfade je OD-Paar entspricht dem etablierten Vorgehen der Hazmat-Literatur, mehrere (auch bewusst unähnliche) Pfade je Relation vorzuhalten und daraus auszuwählen (Akgün, Erkut & Batta, 2000). Zwei **dokumentierte Näherungen** bleiben: (i) die Vorauswahl normiert mit Spaltenmaxima der OD-Tabelle, das MILP mit den Lösungs-Skalen aus §7 — die Auswahl ist also gewichts-aligned, aber nicht skalenidentisch; (ii) sie ist fahrzeugunabhängig, obwohl $c^{km}_v$ fahrzeugspezifisch ist. Die exakte Alternative — alle drei Profile als **Parallelkanten** mit Binärwahl im MILP — verdreifacht die Kantenvariablen und ist als Erweiterung in §12 notiert.

> **Datenvoraussetzung (wichtig für die Interpretation):** $\rho^{\pi} = cost^{\pi} - d^{\pi}$ ist nur einheitenrein, wenn $cost$ je Profil in km-Äquivalenten vorliegt. In der aktuellen OD-Matrix gilt das **nur für das safest-Profil** ($cost/dist \geq 1{,}07$); beim fastest-Profil ist $cost$ eine Zeit in Stunden (Prämie wird auf 0 geklippt), beim shortest-Profil gilt $cost \approx dist$ (Prämie $\approx 0$). Die Mehrprofil-Vorauswahl behandelt fastest/shortest damit systematisch als (scheinbar) risikofrei. **Empfehlung an die Datenpipeline:** je Profil eine eigene, einheitenreine `risk`-Spalte ausgeben (wie in der Charger-Matrix). Bis dahin ist die konservative Alternative, beladene Kanten auf das safest-Profil zu fixieren — im Sinne der in der Hazmat-Literatur empfohlenen konservativen Risikobewertung (Erkut & Verter, 1998; Erkut, Tjandra & Verter, 2007). Der Solver implementiert die Mehrprofil-Auswahl; diese Einschränkung ist bei Ergebnisdiskussion und Sensitivitätsanalysen auszuweisen.

---

## Ebene 2: Mengen (Sets)

| Symbol | Bedeutung | kleine Instanz |
|---|---|---|
| $C$ | Kunden | $\lvert C \rvert = 10$ |
| $0$ | Depot | — |
| $N$ | Routing-Knoten, $N = C \cup \{0\}$ | 11 |
| $V$ | Fahrzeuge (heterogen: Kapazität, Reichweite, Fix-/km-Kosten) | $\lvert V \rvert = 6$ |
| $G_1, \dots, G_m$ | Gruppen **identischer** Fahrzeuge (gleiche Kapazität, Batterie, Reichweite, Fix- und km-Kosten), je Gruppe fest geordnet | aus `vehicles.csv` |
| $S$ | Ladesäulen (HPC), die real angebunden sind | aus Charger-Matrix |
| $P_{side}$ | Side-Trip-Paare $(c,s)$, für die **beide** Richtungen $c \to s$ und $s \to c$ in den Daten existieren; $S(c) = \{ s : (c,s) \in P_{side}\}$ | 8 von 10 Kunden angebunden |
| $A$ | Routing-Kanten: $A = A_{out} \cup A_{ret}$ mit $A_{out} = \{(0,c)\} \cup \{(c,c')\}$ (beladen) und $A_{ret} = \{(c,0)\}$ (leer) | aus Ebene 1b |
| $K$ | Gefahrgutklassen; Prototyp: $K = \{3\}$ (Benzin) | — |

---

## Parameter

| Symbol | Einheit | Wert / Quelle | Bedeutung |
|---|---|---|---|
| $d_{ij}, t_{ij}, \rho_{ij}$ | km, min, km-Äquiv. | Ebene 1b | Distanz, Fahrzeit, Risikoprämie je Routing-Kante; $\rho_{i0} = 0$ auf $A_{ret}$, $\rho = \Lambda$ auf ADR-kritischen Kanten |
| $d^{S}_{cs}, t^{S}_{cs}, \rho^{S}_{cs}$ | km, min, km-Äquiv. | Charger-Matrix | Side-Trip-Kanten, beide Richtungen separat |
| $Dem_c$ | l | Instanz | Nachfrage Kunde $c$ (Summe 57 000 l; $\max_c Dem_c = 9\,500 \leq \min_v Cap_v$) |
| $Cap_v$ | l | `fuel_capacity_l` | Transportkapazität (Kraftstoffmenge) |
| $Range_v$ | km | `range_km` | elektrische Reichweite (SoC-Tracking in km) |
| $F_v$ | EUR | `fixcost` | Fixkosten bei Fahrzeugeinsatz |
| $c^{km}_v$ | EUR/km | `variable_cost_per_km` | variable Kosten |
| $\tau^{serv}$ | min | 45 | Servicezeit je Kunde ($\tau^{serv}_0 = 0$) |
| $\tau^{chg}$ | min | 45 | Ladedauer HPC (zählt als Fahrtunterbrechung) |
| $\tau^{brk}$ | min | 45 | reine Lenkpause (Art. 7 VO (EG) 561/2006) |
| $K^{chg}$ | EUR | 300 | Pauschalkosten je Ladevorgang |
| $\Theta^{blk}$ | min | 270 | max. Lenkzeit am Stück (4,5 h, Art. 7) |
| $\Theta^{day}$ | min | 540 | max. Tageslenkzeit (9 h, Art. 6) |
| $\Theta^{shift}$ | min | 780 | Schichtfenster 13 h $= 24\,\text{h} - 11\,\text{h}$ tägliche Ruhezeit (Art. 8) |
| $w_1, w_2, w_3$ | — | 0,5 / 0,3 / 0,2 | Gewichte Risiko / Kosten / Zeit, $\sum w = 1$; frei variierbar für Pareto-Analysen (Bell, 2006) |
| $M_T, M_D, M_B, M_L$ | — | s. u. | Big-M je Constraint-Typ, bewusst klein gewählt (numerisch günstig, straffere LP-Relaxation) |

$$
M_T = \Theta^{shift} + \tau^{serv} + \max_{(c,s) \in P_{side}}\!\big(t^S_{cs} + t^S_{sc} + \tau^{chg}\big) + \tau^{brk} + \max_{(i,j) \in A} t_{ij}, \qquad
M_D = \Theta^{blk} + \max_{(i,j) \in A} t_{ij}, \qquad
M_B = \max_v Range_v, \qquad
M_L = \max_v Cap_v
$$

---

## Entscheidungsvariablen

| Variable | Typ / Wertebereich | Bedeutung |
|---|---|---|
| $x_{ijv} \in \{0,1\}$ | Binär, $\forall (i,j) \in A, v \in V$ | Fahrzeug $v$ befährt Routing-Kante $(i,j)$ |
| $y_{csv} \in \{0,1\}$ | Binär, $\forall (c,s) \in P_{side}, v$ | $v$ macht vom Kunden $c$ den Lade-Side-Trip zur Säule $s$ (und zurück) |
| $p_{cv} \in \{0,1\}$ | Binär, $\forall c \in C, v$ | $v$ legt am Kunden $c$ eine reine Lenkpause (45 min) ein |
| $T_{nv} \in [0, \Theta^{shift}]$ | stetig | Ankunftszeit an Knoten $n$ (min seit Schichtbeginn) |
| $D^{arr}_{nv}, D^{dep}_{nv} \in [0, \Theta^{blk}]$ | stetig | Lenkzeit seit letzter Unterbrechung bei Ankunft / Abfahrt an $n$ — die Obergrenze $\Theta^{blk}$ ist Teil der Restriktion (Art.-7-Blocklimit) |
| $B^{arr}_{nv}, B^{dep}_{nv} \geq 0$ | stetig | Restreichweite [km] bei Ankunft / Abfahrt an $n$ |
| $L_{nv} \geq 0$ | stetig | Restladung [l] nach Bedienung von Knoten $n$ |

**Abgeleitete Ausdrücke** :

$$
z_v \equiv \sum_{j \in C:\,(0,j) \in A} x_{0jv} \quad \text{(Fahrzeugeinsatz)}, \qquad
visit_{cv} \equiv \sum_{i \in N:\,(i,c) \in A} x_{icv} \quad \text{(Kundenbesuch)}
$$

---

## Zielfunktion 

Drei normierte Kriterien werden als gewichtete Summe minimiert, die Standardskalarisierung des bi-/multikriteriellen Hazmat-Routings (Bell, 2006; Androutsopoulos & Zografos, 2012; Bula et al., 2019):

$$
\min \quad w_1 \frac{R(x,y)}{R_{max}} \;+\; w_2 \frac{K(x,y)}{K_{max}} \;+\; w_3 \frac{Z(x,y,p)}{Z_{max}}
$$

mit

$$
R(x,y) = \sum_{v} \sum_{(i,j) \in A} \rho_{ij}\, x_{ijv} \;+\; \sum_{v} \sum_{(c,s) \in P_{side}} \big(\rho^S_{cs} + \rho^S_{sc}\big)\, y_{csv}
$$

$$
K(x,y) = \sum_{v} F_v\, z_v \;+\; \sum_{v} \sum_{(i,j) \in A} c^{km}_v d_{ij}\, x_{ijv} \;+\; \sum_{v} \sum_{(c,s) \in P_{side}} \Big( c^{km}_v \big(d^S_{cs} + d^S_{sc}\big) + K^{chg} \Big)\, y_{csv}
$$

$$
Z(x,y,p) = \sum_{v} \sum_{(i,j) \in A} t_{ij}\, x_{ijv} \;+\; \sum_{v} \sum_{c \in C} \tau^{serv}\, visit_{cv} \;+\; \sum_{v} \sum_{(c,s) \in P_{side}} \big(t^S_{cs} + t^S_{sc} + \tau^{chg}\big)\, y_{csv} \;+\; \sum_{v} \sum_{c \in C} \tau^{brk}\, p_{cv}
$$

**Eigenschaften.** (i) Rückfahrkanten tragen per Konstruktion $\rho = 0$ bei der Risikoterm wirkt nur auf beladenen Kanten (List & Mirchandani, 1988). (ii) $Z$ ist die **gesamte Einsatzzeit** aller Fahrzeuge (Fahr-, Service-, Lade-, Pausenzeit) $\sum_r (ST^{return} - ST^{start})$, da $T_{0v} = 0$ und keine Wartezeiten modelliert sind. (iii) Alle Kostenterme sind bereits linear in $x, y, z$.

**Normierung** (Worst-Case-Skalen; eine Lösung nutzt höchstens $\lvert C \rvert + \lvert V \rvert$ Routing-Kanten):

$$
R_{max} = \big(\lvert C \rvert + \lvert V \rvert\big) \max_{(i,j)} \rho_{ij}, \qquad
K_{max} = \sum_v F_v + \big(\lvert C \rvert + \lvert V \rvert\big) \max_{(i,j)} d_{ij} \cdot \max_v c^{km}_v + \lvert C \rvert\, K^{chg}, \qquad
Z_{max} = \lvert V \rvert\, \Theta^{shift}
$$

Ohne Normierung wären die Gewichte bedeutungslos, da Risiko (km-Äquivalente), Kosten (EUR) und Zeit (min) um Größenordnungen verschiedene Zahlenbereiche haben, die Notwendigkeit der Normierung bei gewichteter Summe ist ein zentrales Ergebnis von Marler & Arora (2004). Die Variation von $w_1, w_2, w_3$ erzeugt unterschiedliche Punkte der Pareto-Front (Bell, 2006); mit $w_3 = 0$ entsteht das ursprüngliche bikriterielle Risiko-Kosten-Modell.

---

## Nebenbedingungen

### C1: Bedienpflicht
$$
\sum_{v \in V} visit_{cv} = 1 \qquad \forall c \in C
$$
Jeder Kunde wird von genau einem Fahrzeug genau einmal beliefert.

### C2: Flusserhaltung an Kunden
$$
\sum_{i:\,(i,c) \in A} x_{icv} = \sum_{j:\,(c,j) \in A} x_{cjv} \qquad \forall c \in C,\; v \in V
$$
Wer zu einem Kunden fährt, fährt auch wieder ab (Grad-Bedingung des klassischen VRP; Toth & Vigo, 2002).

### C3: Depot-Grad und Single-Trip 
$$
z_v = \sum_{i \in C:\,(i,0) \in A} x_{i0v}, \qquad z_v \leq 1 \qquad \forall v \in V
$$
Anzahl Depot-Abfahrten = Anzahl Depot-Rückkünfte; höchstens **eine Tour je Fahrzeug**.

### C4: Kapazitätsdeckungs-Schnitt 
$$
\sum_{v \in V} Cap_v\, z_v \;\geq\; \sum_{c \in C} Dem_c
$$
Redundant zur Zulässigkeit (folgt aus C1 + C5), aber als klassische Rundungs-/Deckungsungleichung stärkt sie die LP-Relaxation und beschleunigt Branch-and-Bound (Toth & Vigo, 2002).

### C5: Kapazität per Ladungspropagation 
$$
L_{0v} = Cap_v \qquad \forall v; \qquad
L_{jv} \leq L_{iv} - Dem_j + M_L \big(1 - x_{ijv}\big) \qquad \forall (i,j) \in A:\, j \in C,\; \forall v
$$
Das Fahrzeug verlässt das Depot voll beladen; entlang jeder befahrenen Kante sinkt die Restladung um die Nachfrage des Zielkunden. Mit $L \geq 0$ ist die Kapazität implizit eingehalten; zugleich wirkt die Propagation als zusätzliche Subtour-Eliminierung auf beladenen Kanten (kommodifizierte Fluss-/Propagationsformulierungen: Toth & Vigo, 2002).

### C6: Pause/Laden-Exklusivität je Besuch
$$
p_{cv} + \sum_{s \in S(c)} y_{csv} \;\leq\; visit_{cv} \qquad \forall c \in C,\; v \in V
$$
Unterbrechungen sind nur bei tatsächlichem Besuch möglich, und je Kunde höchstens **eine**: entweder reine Lenkpause oder Lade-Side-Trip. Der Ladevorgang (45 min) erfüllt zugleich die vorgeschriebene Fahrtunterbrechung nach Art. 7 VO (EG) 561/2006, die Integration solcher Regelpausen in VRP-Modelle folgt Goel (2009).

### C7: Symmetriebrechung identischer Fahrzeuge 
$$
z_{g,k} \;\geq\; z_{g,k+1} \qquad \forall g \in \{1,\dots,m\},\; k = 1, \dots, \lvert G_g \rvert - 1
$$
Innerhalb jeder Gruppe identischer Fahrzeuge werden Einsätze in fester Reihenfolge aktiviert. Das eliminiert die $\lvert G_g \rvert!$ äquivalenten Permutationen des Suchraums, hierarchische Symmetriebrechungs-Constraints nach Sherali & Smith (2001).

### C8: Zeitpropagation = Subtour-Eliminierung 
$$
T_{0v} = 0; \qquad
T_{jv} \;\geq\; T_{iv} + \tau^{serv}\,\mathbb{1}[i \in C] + E_{iv} + t_{ij} - M_T\big(1 - x_{ijv}\big) \qquad \forall (i,j) \in A:\, j \neq 0,\; \forall v
$$
mit dem Unterbrechungsterm am Vorgängerknoten (nur für $i \in C$):
$$
E_{iv} = \sum_{s \in S(i)} \big(t^S_{is} + t^S_{si} + \tau^{chg}\big)\, y_{isv} + \tau^{brk}\, p_{iv}
$$
Die Ankunftszeit wächst entlang jeder befahrenen Kante strikt (da $t_{ij} > 0$ und $\tau^{serv} > 0$), Zyklen ohne Depot sind damit unmöglich. Die Zeitpropagation ersetzt die MTZ-Familie (Miller, Tucker & Zemlin, 1960) samt $u$-Variablen vollständig; zeitbasierte Formulierungen dieser Art sind als verschärfte MTZ-Varianten etabliert (Desrochers & Laporte, 1991; Kallehauge et al., 2005). Zugleich koppelt C8 Routing und Scheduling in einer Variable  die Untrennbarkeit beider Ebenen im Hazmat-Kontext betonen Bell (2006) und Pradhananga et al. (2014).

### C9: Schichtfenster / Rückkehr
$$
T_{iv} + \tau^{serv} + E_{iv} + t_{i0} \;\leq\; \Theta^{shift} + M_T\big(1 - x_{i0v}\big) \qquad \forall i \in C:\, (i,0) \in A,\; \forall v
$$
Rückkehr am Depot spätestens nach 13 h; zusammen mit der Variablenschranke $T \leq \Theta^{shift}$ liegt die gesamte Schicht im zulässigen Fenster aus Art. 8 VO (EG) 561/2006 (24 h − 11 h tägliche Ruhezeit).

### C10: Lenkzeitfortschreibung 
$$
D^{dep}_{0v} = 0; \qquad
D^{arr}_{jv} \;\geq\; D^{dep}_{iv} + t_{ij} - M_D\big(1 - x_{ijv}\big) \qquad \forall (i,j) \in A,\; \forall v
$$
Die Lenkzeit-Stoppuhr akkumuliert reine Fahrzeit. Die Obergrenze $D^{arr}, D^{dep} \leq \Theta^{blk}$ (Variablenschranke) erzwingt das 4,5-h-Blocklimit nach Art. 7 **einschließlich der Rückfahrkante** $(i,0)$, da die Propagation auch dort gilt (Lenkzeitintegration nach Goel, 2009).

### C11: Lenkzeit-Übergabe am Kunden 
Für alle $c \in C, v \in V$, mit $\chi_{cv} = \sum_{s \in S(c)} y_{csv}$ (Ladeindikator):

$$
\text{(a) Durchreichen ohne Unterbrechung:} \qquad D^{arr}_{cv} - M_D\big(p_{cv} + \chi_{cv}\big) \;\leq\; D^{dep}_{cv} \;\leq\; D^{arr}_{cv} + M_D\big(p_{cv} + \chi_{cv}\big)
$$

$$
\text{(b) Reine Pause: } \quad D^{dep}_{cv} \leq M_D\big(1 - p_{cv}\big) \qquad \Rightarrow \; p_{cv} = 1 \text{ setzt die Uhr auf } 0
$$

$$
\text{(c) Lade-Side-Trip zu } s: \qquad D^{arr}_{cv} + t^S_{cs} \leq \Theta^{blk} + M_D\big(1 - y_{csv}\big), \qquad t^S_{sc} - M_D\big(1 - y_{csv}\big) \leq D^{dep}_{cv} \leq t^S_{sc} + M_D\big(1 - y_{csv}\big)
$$

Fall (c): Die Anfahrt zur Säule muss noch innerhalb des laufenden 4,5-h-Blocks legal sein; nach dem Ladevorgang ($\geq 45$ min ⇒ gilt als Fahrtunterbrechung) zählt nur noch die Rückfahrt von der Säule auf der neuen Uhr. Diese getrennte Ankunfts-/Abfahrts-Buchhaltung ($D^{arr}/D^{dep}$) ist die saubere Verallgemeinerung der Variable $DS$ mit Big-M-Reset.

### C12: Tageslenkzeit 
$$
\sum_{(i,j) \in A} t_{ij}\, x_{ijv} + \sum_{(c,s) \in P_{side}} \big(t^S_{cs} + t^S_{sc}\big)\, y_{csv} \;\leq\; \Theta^{day} \qquad \forall v \in V
$$
Maximal 9 h reine Lenkzeit je Fahrzeug und Tag, inklusive Lade-Abstecher (Art. 6 VO (EG) 561/2006).

### C13: Batterie: Initialisierung und Schranken 
$$
B^{dep}_{0v} = Range_v; \qquad B^{arr}_{nv} \leq Range_v, \quad B^{dep}_{nv} \leq Range_v \qquad \forall n \in N,\; v \in V
$$
Abfahrt am Depot mit voller Batterie (Laden während der Beladung); der Ladezustand wird in km Restreichweite geführt (E-VRP-Standard: Schneider et al., 2014; Erdoğan & Miller-Hooks, 2012).

### C14: Verbrauchspropagation 
$$
B^{arr}_{jv} \;\leq\; B^{dep}_{iv} - d_{ij} + M_B\big(1 - x_{ijv}\big) \qquad \forall (i,j) \in A,\; \forall v
$$
Gilt auch auf Rückfahrkanten mit $B^{arr}_{0v} \geq 0$ ist die Heimkehr-Reichweite gesichert.

### C15: Side-Trip-Reichweite und Ladesemantik 
$$
B^{arr}_{cv} \;\geq\; \sum_{s \in S(c)} d^S_{cs}\, y_{csv} \qquad \text{(Hinfahrt zur Säule machbar)}
$$
$$
B^{dep}_{cv} \;\leq\; B^{arr}_{cv} + M_B\, \chi_{cv} \qquad \text{(ohne Laden: keine Reichweite hinzu)}
$$
$$
B^{dep}_{cv} \;\leq\; Range_v - d^S_{sc} + M_B\big(1 - y_{csv}\big) \qquad \forall s \in S(c) \qquad \text{(mit Laden an } s\text{)}
$$
Nach einem Side-Trip verlässt das Fahrzeug den Kunden mit „vollgeladen minus Rückweg von der Säule". **Dokumentierte Vereinfachung** ggü. Schneider et al. (2014): Ladestationen werden nicht als reguläre Zwischenknoten mit SoC-abhängiger Ladezeit modelliert, sondern als Side-Trips mit konstanter Ladedauer und Vollladung, angemessen, da die Charger-Matrix genau diese Kanten (je Kunde die 3 nächsten Säulen, beide Richtungen) liefert.

---

## Modelltyp und Größenordnung

- **Gemischt-ganzzahliges lineares Programm (MILP)**, zweistufig vorprozessiert (Ebene 1a/1b).
- Problemklasse: **Single-Trip Heterogeneous-Fleet Hazmat-E-VRP mit Lade-Side-Trips und EU-Lenkzeitregeln**  Kombination aus CVRP/HFVRP (Toth & Vigo, 2002; Golden et al., 1984), Hazmat-Risikoziel (Bula et al., 2016; Erkut & Verter, 1998), E-VRP (Schneider et al., 2014; Erdoğan & Miller-Hooks, 2012) und Truck-Driver-Scheduling (Goel, 2009). *Hinweis:* Es handelt sich **nicht** um ein VRPTW — es gibt keine Kunden-Zeitfenster, nur Schicht- und Lenkzeitrestriktionen
- Variablenzahl: $O\big(\lvert A \rvert \cdot \lvert V \rvert\big)$ Binärvariablen für $x$ plus $O\big(\lvert P_{side} \rvert \cdot \lvert V \rvert\big)$ für $y$ und $O\big(\lvert C \rvert \cdot \lvert V \rvert\big)$ für $p$; stetige Variablen $O\big(\lvert N \rvert \cdot \lvert V \rvert\big)$.

---

## Lösungsverfahren

| Baustein | Umsetzung | Begründung / Quelle |
|---|---|---|
| Startlösung | **Warm Start:** Routen einer risikoorientierten Konstruktionsheuristik (JSON) werden als Startwerte der $x$-Variablen gesetzt | Kombination Heuristik + MIP = Matheuristik-Prinzip; eine zulässige Startlösung liefert sofort eine obere Schranke und beschleunigt Branch-and-Bound (Archetti & Speranza, 2014) |
| Exakter Solver | PuLP / **CBC** mit `timeLimit = 600 s`, `gapRel = 2 %`, `warmStart = True` | Androutsopoulos & Zografos (2012): exakte Solver lösen Instanzen dieser Größenordnung in Sekunden bis Minuten; CBC als Open-Source-Alternative benötigt Zeitlimit + Gap-Toleranz, um garantiert eine (nahezu optimale) Lösung zu liefern |
| Optimalitätsnachweis | Auswertung des CBC-Logs (Lower Bound, Gap) | Unterscheidung „optimal" vs. „zulässig bei Zeitlimit" wird im JSON-Export als `search_status` dokumentiert |
| Deutschland-Skala | OR-Tools (CP-VRP-Solver) oder **ALNS** auf derselben Ebene-1-Matrix | Androutsopoulos & Zografos (2012): exakte Solver scheitern bereits bei ~100-Knoten-Netzen; Metaheuristiken erreichen 95–99 % der Lösungsqualität (Bula et al., 2016); für das bikriterielle Hazmat-HFVRP demonstrieren Bula et al. (2019) Nachbarschaftssuche mit Set-Partitioning-Postoptimierung |

---



## Literatur

**Kernliteratur des Projekts:**

- Androutsopoulos, K. N.; Zografos, K. G. (2012): *A bi-objective time-dependent vehicle routing and scheduling problem for hazardous materials distribution.* EURO Journal on Transportation and Logistics 1(1–2), 157–183.
- Bell, M. G. H. (2006): *Mixed Route Strategies for the Risk-Averse Shipment of Hazardous Materials.* Networks and Spatial Economics 6(3–4), 253–265.
- Bula, G. A.; Gonzalez, F. A.; Prodhon, C.; Afsar, H. M.; Velasco, N. M. (2016): *Mixed Integer Linear Programming Model for Vehicle Routing Problem for Hazardous Materials Transportation.* IFAC-PapersOnLine 49(12), 538–543. doi: 10.1016/j.ifacol.2016.07.691
- Cuneo et al. (2018) — Risikogewichtung für die Kraftstofflogistik. *Vermutlich identisch mit:* Carrese, S.; Cuneo, V.; Nigro, M.; Pizzuti, R.; Ardito, C. F.; Marseglia, G.: *Optimization of downstream fuel logistics based on road infrastructure conditions and exposure to accident events.* Transport Policy. doi: 10.1016/j.tranpol.2019.10.016 — bitte mit eurer Quellenliste abgleichen.
- List, G. F.; Mirchandani, P. B. (1988) — Risiko als Funktion der Ladungsmenge; Vorab-Screening ungeeigneter Strecken. (Vollständige bibliografische Angabe gemäß eurer Quellenliste übernehmen.)
- Toth, P.; Vigo, D. (2002): *The Vehicle Routing Problem.* SIAM Monographs on Discrete Mathematics and Applications, Philadelphia.

**Methodische Ergänzungen (belegen einzelne Modellbausteine):**

- Akgün, V.; Erkut, E.; Batta, R. (2000): *On finding dissimilar paths.* European Journal of Operational Research 121(2), 232–246. — Alternativpfad-Menüs je Relation im Hazmat-Routing (→ Ebene 1b).
- Archetti, C.; Speranza, M. G. (2014): *A survey on matheuristics for routing problems.* EURO Journal on Computational Optimization 2(4), 223–246. — Heuristik-Warm-Start für MIP (→ Lösungsverfahren).
- Bula, G. A.; Afsar, H. M.; González, F. A.; Prodhon, C.; Velasco, N. (2019): *Bi-objective vehicle routing problem for hazardous materials transportation.* Journal of Cleaner Production 206, 976–986. doi: 10.1016/j.jclepro.2018.09.228 — bikriterielles Hazmat-HFVRP (Risiko + Kosten), Metaheuristik + Set-Partitioning (→ Zielfunktion, Deutschland-Skala).
- Cattaruzza, D.; Absi, N.; Feillet, D. (2016): *Vehicle routing problems with multiple trips.* 4OR 14(3), 223–259. (→ Ausblick Multi-Trip)
- Crevier, B.; Cordeau, J.-F.; Laporte, G. (2007): *The multi-depot vehicle routing problem with inter-depot routes.* European Journal of Operational Research 176(2), 756–773. (→ Ausblick Multi-Trip, Depot-Kopien)
- Desrochers, M.; Laporte, G. (1991): *Improvements and extensions to the Miller–Tucker–Zemlin subtour elimination constraints.* Operations Research Letters 10(1), 27–36. (→ C8)
- Erdoğan, S.; Miller-Hooks, E. (2012): *A Green Vehicle Routing Problem.* Transportation Research Part E 48(1), 100–114. (→ C13–C15)
- Erkut, E.; Verter, V. (1998): *Modeling of transport risk for hazardous materials.* Operations Research 46(5), 625–642. (→ Ebene 1a, kantenadditive Risikomaße; konservative Risikobewertung)
- Erkut, E.; Tjandra, S. A.; Verter, V. (2007): *Hazardous Materials Transportation.* In: Barnhart, C.; Laporte, G. (Hrsg.): Handbooks in Operations Research and Management Science, Vol. 14, 539–621. (→ Λ-Behandlung gesperrter Relationen)
- Goel, A. (2009): *Vehicle scheduling and routing with drivers' working hours.* Transportation Science 43(1), 17–26. (→ C6, C10–C12)
- Golden, B.; Assad, A.; Levy, L.; Gheysens, F. (1984): *The fleet size and mix vehicle routing problem.* Computers & Operations Research 11(1), 49–66. — HFVRP-Grundlage; die Instanzen dienen Bula et al. (2016) als Basis der Hazmat-Erweiterung.
- Holeczek, N. (2021): *Analysis of different risk models for the hazardous materials vehicle routing problem in urban areas.* Cleaner Environmental Systems 2, 100022. doi: 10.1016/j.cesys.2021.100022 — Pfadgenerierung als Fundament des HMVRP; lastabhängige vs. lastunabhängige Risikomodelle (→ Ebene 1, leere Rückfahrt).
- Kallehauge, B.; Larsen, J.; Madsen, O. B. G.; Solomon, M. M. (2005): *Vehicle Routing Problem with Time Windows.* In: Column Generation, Springer, 67–98. (→ C8, zeitbasierte Formulierung)
- Marler, R. T.; Arora, J. S. (2004): *Survey of multi-objective optimization methods for engineering.* Structural and Multidisciplinary Optimization 26(6), 369–395. (→ Normierung der gewichteten Summe)
- Miller, C. E.; Tucker, A. W.; Zemlin, R. A. (1960): *Integer programming formulation of traveling salesman problems.* Journal of the ACM 7(4), 326–329. (→ klassische MTZ-Referenz, in v3 durch C8 ersetzt)
- Schneider, M.; Stenger, A.; Goeke, D. (2014): *The electric vehicle-routing problem with time windows and recharging stations.* Transportation Science 48(4), 500–520. (→ C13–C15, dokumentierte Side-Trip-Vereinfachung)
- Sherali, H. D.; Smith, J. C. (2001): *Improving discrete model representations via symmetry considerations.* Management Science 47(10), 1396–1407. (→ C7)

**Rechtsquellen:**

- Verordnung (EG) Nr. 561/2006 (Lenk- und Ruhezeiten): Art. 6 (Tageslenkzeit), Art. 7 (Fahrtunterbrechung), Art. 8 (tägliche Ruhezeit → 13-h-Schichtfenster).
- ADR / GGVSEB: Tunnelbeschränkungskategorien je Gefahrgutklasse; ADR 1.1.3.6 (Freistellung kleiner Mengen).
